# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH03/ch03_ART_RULER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About this Notebook

This notebook builds and trains a **math-solving AI agent** using **OpenPipe ART** together with **LangGraph**. The agent solves *Countdown-style arithmetic puzzles*: given a few numbers and a target, it must build a valid arithmetic expression (using +, –, ×, ÷) that hits the exact target.

Here’s what’s going on under the hood:

* **Dataset:** A small dataset of countdown puzzles (5k for training, 100 for testing).
* **Tools:** The agent gets a few specialized tools, one to read the current numbers, one to search for a possible solution, and one to return its final answer.
* **Judge Function:** A deterministic “judge” that checks whether the agent’s math is correct, legal (numbers used only once), and clean (no weird hacks like dividing by zero).
* **Model & Backend:** A local backend with ART and a trainable model (Qwen2.5-7B-Instruct in this example).
* **Rollouts:** The notebook defines how an episode (a “rollout”) runs, the agent gets the puzzle, reasons step-by-step, uses tools, and outputs an expression.
* **Training Loop:** For each batch, the model’s responses are scored in two ways — a hard correctness check, and a soft quality check via **RULER** (a learned evaluator that provides nuanced feedback). The final reward blends both signals to guide training.
* **Testing:** Finally, you test the trained model on unseen puzzles and check whether it solves them correctly.



In [ ]:
# Portions adapted from Unsloth Notebooks
# (https://github.com/unslothai/notebooks)
# and OpenPipe


%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend,langgraph]==0.4.11 langchain-core langgraph langchain_openai tenacity datasets --prerelease allow --no-cache-dir
else:
    try:
        import numpy

        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess

        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend,langgraph]==0.4.11 langchain-core langgraph langchain_openai tenacity datasets protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}

In [ ]:
#!pip install -U openpipe-art[backend,langgraph]

<a name="Environment-Variables"></a>

### Environment Variables

**OpenAI (used for RULER judge model)**

Our RULER reward function queries third-party models to judge the quality of the agent's performance. Any model supported by LiteLLM works. For this example we'll use OpenAI's o4-mini model, so we'll need to set the `OPENAI_API_KEY` environment variable.

**Weights & Biases (optional)**

Later on in the notebook, we'll be creating a model that can automatically logs metrics to Weights & Biases and chat completions to Weave. In order to do so, you'll need to provide your Weights & Biases API key as an environment variable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

warnings.warn("This warning will be hidden")
print("Script continues...")

Script continues...


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
WANDB_API_KEY = os.getenv('WANDB_API_KEY')


In [ ]:
# Clean reinstall of Pillow to resolve 'cannot import name _Ink'
!uv pip uninstall -y pillow pillow-core
!uv pip install --upgrade --force-reinstall "pillow==10.4.0"

import PIL, sys
print("Pillow version:", PIL.__version__)
print(sys.executable)


error: unexpected argument '-y' found

  tip: to pass '-y' as a value, use '-- -y'

Usage: uv pip uninstall [OPTIONS] <PACKAGE|--requirements <REQUIREMENTS>>

For more information, try '--help'.
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 88ms
Prepared 1 package in 88ms
Uninstalled 1 package in 5ms
Installed 1 package in 4ms
 - pillow==12.0.0
 + pillow==10.4.0
Pillow version: 11.3.0
/usr/bin/python3


# Imports


In [ ]:
import re, uuid, ast, operator as op, random, math
from fractions import Fraction
from textwrap import dedent
from typing import List, Optional, Tuple

import pandas as pd
from datasets import load_dataset

import art
from art.local import LocalBackend
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel
from art.langgraph import init_chat_model, wrap_rollout
import weave
from art.rewards import ruler_score_group
from art.utils import iterate_dataset


# Dataset: 5k train / 100 test

In [ ]:
full = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")

random.seed(42)
perm = list(range(len(full)))
random.shuffle(perm)

train_idx = perm[:5000]
test_idx  = perm[5000:5100]  # 100 items

train_ds = full.select(train_idx)
test_ds  = full.select(test_idx)


# Final answer holder

In [ ]:
class FinalAnswer(BaseModel):
    answer: str               # expression string, for example "(44 + 35) + 19"
    source_ids: List[str]     # for bookkeeping

_nonlocal_final = {"value": None}

@tool
def return_final_answer_tool(answer: str, reference_ids: List[str]) -> dict:
    """
    Return final expression string and source ids.
    The expression must evaluate exactly to the target using only allowed numbers at most once each.
    """
    _nonlocal_final["value"] = FinalAnswer(answer=answer, source_ids=reference_ids)
    return _nonlocal_final["value"].model_dump()


# Expression evaluation helpers

In [ ]:
_ALLOWED_BIN = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv}
_ALLOWED_UN  = {ast.USub: op.neg, ast.UAdd: op.pos}

def _eval_ast(node):
    """Evaluate with float for permissive mode."""
    if isinstance(node, ast.Expression):
        return _eval_ast(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if hasattr(ast, "Num") and isinstance(node, ast.Num):
        return float(node.n)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UN:
        return _ALLOWED_UN[type(node.op)](_eval_ast(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BIN:
        left = _eval_ast(node.left)
        right = _eval_ast(node.right)
        if isinstance(node.op, ast.Div) and right == 0:
            raise ZeroDivisionError
        return _ALLOWED_BIN[type(node.op)](left, right)
    raise ValueError("Unsupported syntax node")

def _eval_ast_fraction(node) -> Fraction:
    """Evaluate with Fraction for strict integer intermediate checks."""
    if isinstance(node, ast.Expression):
        return _eval_ast_fraction(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return Fraction(node.value).limit_denominator()
    if hasattr(ast, "Num") and isinstance(node, ast.Num):
        return Fraction(node.n).limit_denominator()
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UN:
        return _ALLOWED_UN[type(node.op)](_eval_ast_fraction(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BIN:
        left = _eval_ast_fraction(node.left)
        right = _eval_ast_fraction(node.right)
        if isinstance(node.op, ast.Div) and right == 0:
            raise ZeroDivisionError
        return _ALLOWED_BIN[type(node.op)](left, right)
    raise ValueError("Unsupported syntax node")

def safe_eval_expr(expr: str) -> float:
    tree = ast.parse(expr, mode="eval")
    return float(_eval_ast(tree))

def numbers_used_in_expr(expr: str) -> List[int]:
    """
    Return list of integer literals used in the expression.
    Counts only true integer literals. Treats unary negatives as the same by abs().
    """
    tree = ast.parse(expr, mode="eval")
    used: List[int] = []

    class _Collector(ast.NodeVisitor):
        def visit_Constant(self, node: ast.Constant):
            if isinstance(node.value, (int, float)):
                v = float(node.value)
                if v.is_integer():
                    used.append(abs(int(v)))

        def visit_UnaryOp(self, node: ast.UnaryOp):
            self.visit(node.operand)

    _Collector().visit(tree)
    return used

# Tools for the agent

In [ ]:
class CountdownItem(BaseModel):
    target: int
    nums: List[int]
    id: str

_current_scenario = {"target": None, "nums": None, "id": None}

@tool
def read_countdown_context() -> dict:
    """Return the active countdown numbers and target as {'nums': [...], 'target': int}."""
    return {"nums": _current_scenario["nums"], "target": _current_scenario["target"]}


# Deterministic judge

In [ ]:
def judge_countdown_expression(
    target: int,
    allowed_nums: List[int],
    expr: str,
    enforce_integer_intermediates: bool = False,
) -> Tuple[float, Optional[float], Optional[str]]:
    """
    Returns (reward, value, reason).
    Reward is 1.0 if the expression evaluates to target and uses the allowed numbers legally, else 0.0.
    If enforce_integer_intermediates is true, evaluation uses Fraction.
    """
    expr = expr.strip()
    if not expr:
        return 0.0, None, "empty expression"

    # Validate characters to be safe
    if re.search(r"[^0-9\+\-\*\/\(\)\.\s]", expr):
        return 0.0, None, "invalid characters"

    # Check number usage
    used = numbers_used_in_expr(expr)

    def counts(xs):
        d = {}
        for v in xs:
            d[v] = d.get(v, 0) + 1
        return d

    allowed_counts = counts([int(x) for x in allowed_nums])
    used_counts    = counts(used)

    for v, c in used_counts.items():
        if v not in allowed_counts or c > allowed_counts[v]:
            return 0.0, None, f"illegal number usage: {v}"

    try:
        if enforce_integer_intermediates:
            val_frac = _eval_ast_fraction(ast.parse(expr, mode="eval"))
            if val_frac == Fraction(target, 1):
                return 1.0, float(val_frac), None
            return 0.0, float(val_frac), "wrong value"
        else:
            val = safe_eval_expr(expr)
            if abs(val - target) < 1e-9:
                return 1.0, val, None
            return 0.0, val, "wrong value"
    except ZeroDivisionError:
        return 0.0, None, "division by zero"
    except Exception as e:
        return 0.0, None, f"eval error: {e}"

# Model and backend

In [ ]:
random.seed(42)

model = art.TrainableModel(
    name="countdown-agent-001",
    project="countdown-agent",
    base_model="Qwen/Qwen2.5-7B-Instruct",
)


model._internal_config = art.dev.InternalModelConfig(
    init_args=art.dev.InitArgs(max_seq_length=4096),
    engine_args=art.dev.EngineArgs(enforce_eager=True, gpu_memory_utilization=0.8),
)

backend = LocalBackend(in_process=True, path="./.art")
await model.register(backend)


# Scenarios

In [ ]:
class Scenario(BaseModel):
    id: str
    target: int
    nums: List[int]
    question: str

def mk_scenario(row, idx: int) -> Scenario:
    tgt  = int(row["target"])
    nums = [int(x) for x in row["nums"]]
    q = (
        f"Using only the numbers {nums} each at most once, write a valid arithmetic expression "
        f"with +, -, *, or / that evaluates exactly to {tgt}. Return only the expression string."
    )
    return Scenario(id=f"cd_{idx}", target=tgt, nums=nums, question=q)

train_scenarios = [mk_scenario(train_ds[i], i) for i in range(len(train_ds))]
test_scenarios  = [mk_scenario(test_ds[i], i)  for i in range(len(test_ds))]


# Exact search tool with Fractions

In [ ]:
@tool
def countdown_search_tool(
    nums: List[int],
    target: int,
    limit_nodes: int = 5000,
    require_integer_intermediates: bool = True
) -> str:
    """
    Try to find a valid expression using each number at most once with + - * /.
    Uses Fraction to avoid numeric error and can require integer intermediates.
    Returns an expression string or "" if none found within the limit.
    """
    def ok_div(a: Fraction, b: Fraction) -> bool:
        if b == 0:
            return False
        return True if not require_integer_intermediates else (a % b == 0)

    ops = [
        ("+", lambda a,b,a_s,b_s: (a + b, f"({a_s}+{b_s})", True)),
        ("*", lambda a,b,a_s,b_s: (a * b, f"({a_s}*{b_s})", True)),
        ("-", lambda a,b,a_s,b_s: (a - b, f"({a_s}-{b_s})", True)),
        ("/", lambda a,b,a_s,b_s: (a / b, f"({a_s}/{b_s})", ok_div(a,b))),
    ]

    start = tuple((Fraction(n, 1), str(n)) for n in nums)
    seen = set()
    nodes = 0

    def key(state):
        # multiset of values, order free
        return tuple(sorted((v.numerator, v.denominator) for v,_ in state))

    def search(state):
        nonlocal nodes
        nodes += 1
        if nodes > limit_nodes:
            return ""
        if len(state) == 1:
            v, s = state[0]
            return s if v == Fraction(target, 1) else ""
        k = key(state)
        if k in seen:
            return ""
        seen.add(k)

        n = len(state)
        for i in range(n):
            for j in range(i+1, n):
                a, a_s = state[i]
                b, b_s = state[j]
                rest = tuple(state[k] for k in range(n) if k not in (i, j))

                # commutative first
                for sym, fn in ops:
                    if sym in {"+", "*"}:
                        v, s, good = fn(a, b, a_s, b_s)
                        if good:
                            res = search(rest + ((v, s),))
                            if res:
                                return res
                    else:
                        v1, s1, good1 = fn(a, b, a_s, b_s)
                        if good1:
                            res = search(rest + ((v1, s1),))
                            if res:
                                return res
                        v2, s2, good2 = fn(b, a, b_s, a_s)
                        if good2:
                            res = search(rest + ((v2, s2),))
                            if res:
                                return res
        return ""

    return search(start)


# Rollout

In [ ]:
MAX_TURNS = 8

class ProjectTrajectory(art.Trajectory):
    final_answer: Optional[FinalAnswer] = None

class CountdownScenario(BaseModel):
    step: int
    scenario: Scenario

@weave.op
async def rollout(model: art.Model, cd: CountdownScenario) -> ProjectTrajectory:
    scn = cd.scenario

    _nonlocal_final["value"] = None
    _current_scenario["target"] = scn.target
    _current_scenario["nums"]   = scn.nums
    _current_scenario["id"]     = scn.id

    nums_str = ",".join(map(str, scn.nums))
    traj = ProjectTrajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={"scenario_id": scn.id, "target": float(scn.target), "nums": nums_str},
    )
    if traj.metrics is None:
        traj.metrics = {}

    system_prompt = dedent(f"""
        You are a math agent for Countdown tasks.

        Goal: produce one expression that evaluates exactly to {scn.target}
        using only the numbers {scn.nums} each at most once. Operators: +, -, *, /.
        Parentheses are allowed.

        Tools:
        - read_countdown_context()
        - countdown_search_tool(nums=[...], target=..., require_integer_intermediates=True)
        - return_final_answer_tool(answer=EXPR, reference_ids=[...])

        Process:
        1) Call read_countdown_context to confirm inputs.
        2) Call countdown_search_tool. If it returns a non empty expression,
           pass it verbatim to return_final_answer_tool.
        3) If it returns empty, reason and still obey number usage.
        The final tool must receive ONLY the expression string.
    """)

    tools = [read_countdown_context, countdown_search_tool, return_final_answer_tool]
    chat  = init_chat_model(model.name, temperature=0.6, top_p=0.9, max_tokens=256)
    agent = create_react_agent(chat, tools)

    try:
        config = {"configurable": {"thread_id": str(uuid.uuid4()),
                                   "seed": random.randint(0, 1_000_000)},
                                    "recursion_limit": MAX_TURNS}
        await agent.ainvoke(
            {"messages": [SystemMessage(content=system_prompt), HumanMessage(content=scn.question)]},
            config=config,
        )

        if _nonlocal_final["value"]:
            traj.final_answer = _nonlocal_final["value"]
            reward, val, reason = judge_countdown_expression(
                scn.target, scn.nums, traj.final_answer.answer, enforce_integer_intermediates=True
            )
            traj.reward = float(reward)

            # numeric only metrics
            traj.metrics["correct"] = float(reward)
            traj.metrics["value"] = float(val) if isinstance(val, (int, float)) else float("nan")

            # text info into metadata
            traj.metadata["last_reason"] = str(reason) if reason else ""

            # rubric for RULER
            traj.metadata["rubric"] = (
                "Score higher if the expression (1) hits the target, "
                "(2) uses each provided number at most once, "
                "(3) uses fewer operations, "
                "(4) avoids redundant parentheses and trivial +/-0 or *1 tricks, "
                "(5) avoids division when +, -, * suffice."
            )
    except Exception as e:
        print("Rollout error:", e)
        traj.messages_and_choices.append({"role": "assistant", "content": f"Error: {e}"})
    return traj

# Utils for summaries

In [ ]:
def groups_to_df(groups: list[art.TrajectoryGroup], label: str) -> pd.DataFrame:
    rows = []
    for gi, g in enumerate(groups):
        for ti, t in enumerate(g.trajectories):
            rows.append({
                "group": gi,
                "traj": ti,
                "reward": getattr(t, "reward", None),
                "correct": t.metrics.get("correct") if hasattr(t, "metrics") else None,
                "reason": (getattr(t, "metadata", {}) or {}).get("last_reason", ""),
                "expr": getattr(getattr(t, "final_answer", None), "answer", None),
                "steps": len(getattr(t, "messages_and_choices", [])),
            })
    df = pd.DataFrame(rows)
    df.attrs["label"] = label
    return df

def print_group_summary(groups, title):
    print(f"\n=== {title} ===")
    df = groups_to_df(groups, title)
    if len(df):
        print(df.to_string(index=False))
    else:
        print("(no groups)")

# Training loop & Inference

In [ ]:
training_config = {
    "groups_per_step": 4,
    "num_epochs": 1,
    "rollouts_per_group": 2,
    "learning_rate": 1e-5,
    "max_steps": 3,
}

train_slice = train_scenarios[:64]

training_iterator = iterate_dataset(
    train_slice,
    groups_per_step=training_config["groups_per_step"],
    num_epochs=training_config["num_epochs"],
    initial_step=await model.get_step(),
)

for batch in training_iterator:
    print(f"Training step {batch.step}, epoch {batch.epoch}")
    groups = []
    for scn in batch.items:
        group = art.TrajectoryGroup(
            [ wrap_rollout(model, rollout)(model, CountdownScenario(step=batch.step, scenario=scn))
              for _ in range(training_config["rollouts_per_group"]) ]
        )
        groups.append(group)

    finished = await art.gather_trajectory_groups(
        groups, pbar_desc="gather",
        max_exceptions=training_config["rollouts_per_group"] * len(batch.items),
    )

    print_group_summary(finished, "Pre judge group summary")

    # Hard correctness gate already set reward and metrics["correct"]
    judged = finished

    print_group_summary(judged, "Post judge group summary")

    # Ensure numeric only metrics exist
    def _sanitize(groups):
        for g in groups:
            for t in g.trajectories:
                if t.metrics is None:
                    t.metrics = {}
                t.metrics["correct"] = float(t.metrics.get("correct", 0.0))
                v = t.metrics.get("value", math.nan)
                try:
                    t.metrics["value"] = float(v)
                except Exception:
                    t.metrics["value"] = math.nan
                if t.metadata is None:
                    t.metadata = {}
        return groups

    judged = _sanitize(judged)

    # Ask RULER to score each group. Guard on errors or availability.
    ruler_model_id = "openai/gpt-4.1"  # change to a model you have
    ruler_groups = []
    for group in judged:
        try:
            rg = await ruler_score_group(group, ruler_model_id, debug=True)
        except Exception:
            rg = None
        ruler_groups.append(rg if rg is not None else group)

    # Combine: final = gate * (alpha + beta * ruler_norm)
    alpha, beta = 0.7, 0.3

    def _combine(groups, alpha: float = 1.0, beta: float = 1.0):
        for g in groups:
            raw = []
            for t in g.trajectories:
                try:
                    raw.append(float(t.reward))
                except Exception:
                    raw.append(0.0)

            rmin = min(raw) if raw else 0.0
            rmax = max(raw) if raw else 0.0
            span = (rmax - rmin) if (rmax > rmin) else 1.0

            for t in g.trajectories:
                if t.metrics is None:
                    t.metrics = {}
                if t.metadata is None:
                    t.metadata = {}

                gate = float(t.metrics.get("correct", 0.0) or 0.0)
                gate = 1.0 if gate >= 0.5 else 0.0

                try:
                    ruler_raw = float(t.reward)
                except Exception:
                    ruler_raw = 0.0
                ruler_norm = (ruler_raw - rmin) / span
                ruler_norm = max(0.0, min(1.0, ruler_norm)) if ruler_norm == ruler_norm else 0.0

                final_reward = gate * (alpha + beta * ruler_norm)
                final_reward = max(0.0, min(1.0, final_reward))

                t.metadata["ruler_score_raw"] = float(ruler_raw)
                t.metrics["ruler_norm"] = float(ruler_norm)
                t.metrics["final_reward"] = float(final_reward)
                t.reward = float(final_reward)
        return groups

    hybrid_groups = _combine(ruler_groups, alpha=alpha, beta=beta)

    print_group_summary(hybrid_groups, "Hybrid (gate + RULER) summary")

    await model.train(
        hybrid_groups,
        config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
        _config={"logprob_calculation_chunk_size": 8},
    )

    print(f"Completed training step {batch.step}")
    if batch.step >= training_config["max_steps"]:
        break


# Inference on 10 held out tests

print("\nTesting trained model on 10 held out tasks...\n")
for scn in test_scenarios[:10]:
    res = await wrap_rollout(model, rollout)(model, CountdownScenario(step=0, scenario=scn))
    expr = res.final_answer.answer if res.final_answer else None
    reward, val, reason = judge_countdown_expression(scn.target, scn.nums, expr or "", enforce_integer_intermediates=True)
    status = "CORRECT" if reward == 1.0 else f"WRONG ({reason})"
    print(f"nums={scn.nums} target={scn.target} -> {expr} => {status}")


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.85M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

INFO 10-23 07:28:49 [__init__.py:235] Automatically detected platform cuda.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.8.6: Fast Qwen2 patching. Transformers: 4.53.2. vLLM: 0.10.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 78.2%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 22.16 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 409

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!
Completed training step 2
Training step 3, epoch 0


gather:   0%|          | 0/8 [00:00<?, ?it/s]


=== Pre judge group summary ===
 group  traj  reward  correct                   reason              expr  steps
     0     0     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      7
     0     1     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      7
     1     0     1.0      1.0                          (35-((84-30)/27))      7
     1     1     1.0      1.0                          (35-((84-30)/27))      7
     2     0     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      9
     2     1     1.0      1.0                             67 - (33 - 33)      9
     3     0     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      7
     3     1     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      7

=== Post judge group summary ===
 group  traj  reward  correct                   reason              expr  steps
     0     0     0.0      0.0 illegal number usage: 35 (35-((84-30)/27))      7
     0     1     0.0      0.0 illegal number usage: 3

train:   0%|          | 0/1 [00:00<?, ?it/s]

Completed training step 3

Testing trained model on 10 held out tasks...

nums=[22, 89, 16] target=51 -> None => WRONG (empty expression)
nums=[85, 45, 75, 10] target=25 -> ((75-10)-(85-45)) => CORRECT
nums=[68, 96, 50, 3] target=75 -> ((50-3)-(68-96)) => CORRECT
nums=[49, 94, 73, 40] target=12 -> ((40-73)-(49-94)) => CORRECT
nums=[40, 79, 73] target=34 -> (73+(40-79)) => CORRECT
nums=[21, 1, 33, 2] target=40 -> (33+(21/(1+2))) => CORRECT
nums=[29, 1, 4, 46] target=66 -> ((4*(29-1))-46) => CORRECT
nums=[19, 97, 98] target=18 -> ((19+97)-98) => CORRECT
nums=[55, 76, 1] target=22 -> (1-(55-76)) => CORRECT
nums=[31, 4, 16, 36] target=20 -> None => WRONG (empty expression)
